# Notebook 02 — Feature Engineering
Defines the feature extractor, fits the TF-IDF vectorizer, and exports everything to `src/feature_extractor.py`.

## Cell 1 — Imports

Loads all dependencies required for feature extraction and vectorisation,
including pattern matching utilities, numerical computing libraries, the dataset
reader, the serialisation library for saving model artefacts, and the TF-IDF
vectoriser from scikit-learn.

The outputs directory is created at this stage if it does not already exist.

In [1]:
# Cell 1 — Imports
import re
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer

# Paths
DATA_CSV       = os.path.join('..', 'data', 'emails.csv')
VECTORIZER_PKL = os.path.join('..', 'outputs', 'vectorizer.pkl')
os.makedirs(os.path.join('..', 'outputs'), exist_ok=True)

print('Imports OK.')

Imports OK.


## Cell 2 — `extract_manual_features(text)`

This cell defines the hand-crafted feature extraction function, which produces
eight interpretable numerical signals from a raw email string. These features
are strong phishing emails signals and complement the
statistical TF-IDF representation defined in later cells.

| Feature | What it captures |
|---------|------------------|
| `url_count` | Total number of URLs (http/https links) in the email |
| `has_http_only` | 1 if any URL uses plain `http://` (not encrypted) |
| `suspicious_url_keywords` | 1 if any URL contains phishing keywords (login, verify, account …) |
| `urgent_keyword_count` | Count of urgency/scare phrases like 'click here', 'suspended', 'urgent' |
| `has_ip_url` | 1 if a URL uses a raw IP address instead of a domain name |
| `exclamation_count` | Number of `!` characters (phishing emails overuse them) |
| `has_form_tag` | 1 if the email HTML contains `<form>` or `<input>` tags |
| `link_to_text_ratio` | `url_count / word_count` — high ratio means mostly links, little real text |

In [2]:
# Cell 2 — extract_manual_features(text)

# Compiled regex patterns (defined once at module level for speed)
_URL_PATTERN = re.compile(
    r'https?://[^\s<>"]+|www\.[^\s<>"]+',
    re.IGNORECASE
)
_IP_URL_PATTERN = re.compile(
    r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}',
    re.IGNORECASE
)
_SUSPICIOUS_KEYWORDS = [
    'login', 'verify', 'account', 'secure', 'update', 'confirm', 'banking'
]
_URGENT_PHRASES = [
    'urgent', 'click here', 'suspended', 'verify your account',
    'limited time', 'act now', 'immediately', 'your account has been',
    'confirm your', 'update your', 'expires soon', 'unusual activity'
]
_FORM_TAG_PATTERN = re.compile(r'<\s*(form|input)[\s>]', re.IGNORECASE)


def extract_manual_features(text: str) -> dict:
    """
    Extract 8 hand-crafted phishing-signal features from an email string.

    Parameters
    ----------
    text : str  — raw or cleaned email text

    Returns
    -------
    dict with keys:
        url_count, has_http_only, suspicious_url_keywords,
        urgent_keyword_count, has_ip_url, exclamation_count,
        has_form_tag, link_to_text_ratio
    """
    text_lower = text.lower()
    urls       = _URL_PATTERN.findall(text)

    # 1. url_count
    url_count = len(urls)

    # 2. has_http_only — any URL is plain http (not https)
    has_http_only = int(
        any(u.lower().startswith('http://') for u in urls)
    )

    # 3. suspicious_url_keywords — phishing keyword appears inside a URL
    suspicious_url_keywords = int(
        any(
            kw in u.lower()
            for u in urls
            for kw in _SUSPICIOUS_KEYWORDS
        )
    )

    # 4. urgent_keyword_count — total occurrences of urgency phrases
    urgent_keyword_count = sum(
        text_lower.count(phrase) for phrase in _URGENT_PHRASES
    )

    # 5. has_ip_url — URL uses raw IP address instead of domain name
    has_ip_url = int(bool(_IP_URL_PATTERN.search(text)))

    # 6. exclamation_count — number of '!' characters
    exclamation_count = text.count('!')

    # 7. has_form_tag — HTML <form> or <input> present
    has_form_tag = int(bool(_FORM_TAG_PATTERN.search(text)))

    # 8. link_to_text_ratio — url_count / word_count
    word_count = len(text.split())
    link_to_text_ratio = url_count / word_count if word_count > 0 else 0.0

    return {
        'url_count':               url_count,
        'has_http_only':           has_http_only,
        'suspicious_url_keywords': suspicious_url_keywords,
        'urgent_keyword_count':    urgent_keyword_count,
        'has_ip_url':              has_ip_url,
        'exclamation_count':       exclamation_count,
        'has_form_tag':            has_form_tag,
        'link_to_text_ratio':      round(link_to_text_ratio, 4),
    }


# Quick sanity check
sample = 'URGENT! Verify your account now at http://192.168.1.1/login?verify=true !!!'
print(extract_manual_features(sample))

{'url_count': 1, 'has_http_only': 1, 'suspicious_url_keywords': 1, 'urgent_keyword_count': 2, 'has_ip_url': 1, 'exclamation_count': 4, 'has_form_tag': 0, 'link_to_text_ratio': 0.125}


## Cell 3 — TF-IDF vectoriser: fit

Defines `fit_vectorizer()`, which trains a TF-IDF vectoriser on the full email
corpus. The vectoriser builds a vocabulary of the 3,000 most informative terms,
excluding common stop words that carry no discriminative value.

The fitted vectoriser is serialised to `outputs/vectorizer.pkl` using joblib.
This file is a shared project artefact — it must be present before any other
notebook or the Flask backend attempts to transform email text into feature
vectors.


In [3]:
# Cell 3 — TF-IDF: fit_vectorizer(texts)

def fit_vectorizer(texts: list) -> TfidfVectorizer:
    """
    Fit a TF-IDF vectorizer on a list of email strings.
    Saves the fitted vectorizer to outputs/vectorizer.pkl.

    Parameters
    ----------
    texts : list of str

    Returns
    -------
    Fitted TfidfVectorizer
    """
    vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
    vectorizer.fit(texts)
    joblib.dump(vectorizer, VECTORIZER_PKL)
    print(f'Vectorizer fitted on {len(texts)} texts.')
    print(f'Vocabulary size : {len(vectorizer.vocabulary_)}')
    print(f'Saved           → {VECTORIZER_PKL}')
    return vectorizer

print('fit_vectorizer() defined.')

fit_vectorizer() defined.


## Cell 4 — TF-IDF vectoriser: transform

Defines `transform_text()`, which converts a single email string into a dense
numerical vector using the fitted vocabulary saved in Cell 3.

This function is designed for inference — it is called once per email at
prediction time and produces a vector of 3,000 values that reflects the
relative importance of each vocabulary term in the input text.

In [4]:
# Cell 4 — TF-IDF: transform_text(text)

def transform_text(text: str) -> np.ndarray:
    """
    Transform a single email string into a dense TF-IDF vector.
    Loads the fitted vectorizer from outputs/vectorizer.pkl.

    Parameters
    ----------
    text : str

    Returns
    -------
    np.ndarray of shape (3000,)
    """
    vectorizer = joblib.load(VECTORIZER_PKL)
    return vectorizer.transform([text]).toarray()[0]

print('transform_text() defined.')

transform_text() defined.


## Cell 5 — Combined feature extraction

Defines `extract_features()` — the primary public interface of this module and
the single function imported by all other project components.

It calls the manual feature function from Cell 2 and the TF-IDF transform from
Cell 4, then concatenates their outputs into one unified vector. This design
keeps both feature types — interpretable signals and statistical word patterns —
in a single, consistent representation.

| Return value | Shape | Used by |
|---|---|---|
| Combined feature vector | (3,008,) | Member 2 for model training; Member 3 for inference |
| Manual features dictionary | 8 keys | Flask backend — rendered as indicators on the results page |

In [5]:
# Cell 5 — extract_features(text) — Combined Export

def extract_features(text: str):
    """
    Main feature extraction function used by all notebooks and Flask.

    Combines:
      - 8 manual features  (extract_manual_features)
      - 3000-dim TF-IDF vector  (transform_text)

    Parameters
    ----------
    text : str — raw email text

    Returns
    -------
    combined_vector : np.ndarray of shape (3008,)
    manual_dict     : dict with the 8 interpretable features
    """
    manual_dict   = extract_manual_features(text)
    manual_vector = np.array(list(manual_dict.values()), dtype=np.float64)
    tfidf_vector  = transform_text(text)                # shape (3000,)
    combined      = np.concatenate([manual_vector, tfidf_vector])  # (3008,)
    return combined, manual_dict

print('extract_features() defined.')

extract_features() defined.


## Cell 6 — Fit vectoriser on full corpus

Executes `fit_vectorizer()` against the complete training dataset loaded from
`data/emails.csv`. This is the only cell in the notebook that writes a file
to disk as a side effect.

All subsequent calls to `transform_text()` and `extract_features()` — in this
notebook, in Member 2's training notebook, and in the Flask backend — depend on
the vocabulary file produced here.


In [6]:
# Cell 6 — Fit Vectorizer on Dataset

df = pd.read_csv(DATA_CSV)
print(f'Loaded {DATA_CSV}  →  {df.shape}')

fit_vectorizer(df['text'].tolist())   # produces outputs/vectorizer.pkl
print('\nvectorizer.pkl is ready.')

Loaded ..\data\emails.csv  →  (3882, 2)
Vectorizer fitted on 3882 texts.
Vocabulary size : 3000
Saved           → ..\outputs\vectorizer.pkl

vectorizer.pkl is ready.


## Cell 7 — Validation tests

Runs `extract_features()` against three representative inputs to verify
end-to-end pipeline correctness before the module is exported for team use.

| Test case | Purpose |
|-----------|---------|
| Phishing email | Confirms that high-risk signals produce non-zero manual feature values |
| Legitimate email | Confirms that low-risk content produces appropriately low feature values |
| Empty string | Confirms graceful handling of edge-case input without errors |

Two assertions are enforced for each test case:

- Output shape is exactly `(3,008,)` across all inputs.
- No NaN or missing values are present in any output vector.

The manual feature dictionary is printed for each case to allow
human inspection of individual feature values prior to export.

In [7]:
# Cell 7 — Validation Tests

PHISHING_SAMPLE = (
    'URGENT! Your PayPal account has been suspended. '
    'Click here to verify your account immediately: '
    'http://192.168.0.1/login?confirm=true&account=update !!!'
)
HAM_SAMPLE = (
    'Hi John, just a reminder about our team meeting tomorrow at 10am. '
    'Please review the attached agenda and let me know if you have questions.'
)
EDGE_SAMPLE = ''   # empty email edge case

for name, email in [('Phishing', PHISHING_SAMPLE), ('Ham', HAM_SAMPLE), ('Edge (empty)', EDGE_SAMPLE)]:
    vec, feats = extract_features(email)
    print(f'\n[{name}]')
    print(f'  Vector shape : {vec.shape}')
    print(f'  NaN count    : {np.isnan(vec).sum()}')
    print(f'  Manual feats : {feats}')
    # Assertions
    assert vec.shape == (3008,), f'Expected (3008,), got {vec.shape}'
    assert np.isnan(vec).sum() == 0, 'NaN values detected!'

print('\n✓ All validation tests passed.')


[Phishing]
  Vector shape : (3008,)
  NaN count    : 0
  Manual feats : {'url_count': 1, 'has_http_only': 1, 'suspicious_url_keywords': 1, 'urgent_keyword_count': 5, 'has_ip_url': 1, 'exclamation_count': 4, 'has_form_tag': 0, 'link_to_text_ratio': 0.0625}

[Ham]
  Vector shape : (3008,)
  NaN count    : 0
  Manual feats : {'url_count': 0, 'has_http_only': 0, 'suspicious_url_keywords': 0, 'urgent_keyword_count': 0, 'has_ip_url': 0, 'exclamation_count': 0, 'has_form_tag': 0, 'link_to_text_ratio': 0.0}

[Edge (empty)]
  Vector shape : (3008,)
  NaN count    : 0
  Manual feats : {'url_count': 0, 'has_http_only': 0, 'suspicious_url_keywords': 0, 'urgent_keyword_count': 0, 'has_ip_url': 0, 'exclamation_count': 0, 'has_form_tag': 0, 'link_to_text_ratio': 0.0}

✓ All validation tests passed.


## Cell 8 — Export to `src/feature_extractor.py`

Writes the finalised module to `src/feature_extractor.py` using the
`%%writefile` cell magic. The exported file contains all four functions:
`extract_manual_features`, `fit_vectorizer`, `transform_text`, and
`extract_features`.

Path resolution in the exported module is handled relative to the file's own
location, ensuring it can be imported correctly regardless of the working
directory — whether from another notebook, a test runner, or the Flask server.


In [8]:
%%writefile ../src/feature_extractor.py
"""
src/feature_extractor.py
Shared module — generated by notebooks/02_feature_engineering.ipynb
Do not edit manually; re-run Cell 8 to regenerate.
"""
import re
import os
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

VECTORIZER_PKL = os.path.join(os.path.dirname(__file__), '..', 'outputs', 'vectorizer.pkl')

_URL_PATTERN = re.compile(r'https?://[^\s<>"]+|www\.[^\s<>"]+', re.IGNORECASE)
_IP_URL_PATTERN = re.compile(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', re.IGNORECASE)
_SUSPICIOUS_KEYWORDS = ['login', 'verify', 'account', 'secure', 'update', 'confirm', 'banking']
_URGENT_PHRASES = [
    'urgent', 'click here', 'suspended', 'verify your account',
    'limited time', 'act now', 'immediately', 'your account has been',
    'confirm your', 'update your', 'expires soon', 'unusual activity'
]
_FORM_TAG_PATTERN = re.compile(r'<\s*(form|input)[\s>]', re.IGNORECASE)


def extract_manual_features(text: str) -> dict:
    text_lower = text.lower()
    urls = _URL_PATTERN.findall(text)
    url_count = len(urls)
    has_http_only = int(any(u.lower().startswith('http://') for u in urls))
    suspicious_url_keywords = int(
        any(kw in u.lower() for u in urls for kw in _SUSPICIOUS_KEYWORDS)
    )
    urgent_keyword_count = sum(text_lower.count(p) for p in _URGENT_PHRASES)
    has_ip_url = int(bool(_IP_URL_PATTERN.search(text)))
    exclamation_count = text.count('!')
    has_form_tag = int(bool(_FORM_TAG_PATTERN.search(text)))
    word_count = len(text.split())
    link_to_text_ratio = round(url_count / word_count, 4) if word_count > 0 else 0.0
    return {
        'url_count': url_count,
        'has_http_only': has_http_only,
        'suspicious_url_keywords': suspicious_url_keywords,
        'urgent_keyword_count': urgent_keyword_count,
        'has_ip_url': has_ip_url,
        'exclamation_count': exclamation_count,
        'has_form_tag': has_form_tag,
        'link_to_text_ratio': link_to_text_ratio,
    }


def fit_vectorizer(texts: list) -> TfidfVectorizer:
    vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
    vectorizer.fit(texts)
    joblib.dump(vectorizer, VECTORIZER_PKL)
    return vectorizer


def transform_text(text: str) -> np.ndarray:
    vectorizer = joblib.load(VECTORIZER_PKL)
    return vectorizer.transform([text]).toarray()[0]


def extract_features(text: str):
    manual_dict   = extract_manual_features(text)
    manual_vector = np.array(list(manual_dict.values()), dtype=np.float64)
    tfidf_vector  = transform_text(text)
    combined      = np.concatenate([manual_vector, tfidf_vector])
    return combined, manual_dict

Overwriting ../src/feature_extractor.py
